In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
def get_timings():
    data = []
    for dataset in Path("../configs/dataset").glob("*.yaml"):
        dataset_name, size, seed = dataset.name.split("_")
        size = int(size)
        seed = int(seed.split(".")[0])
        experiment_path = Path(f"../experiments/{dataset_name}_{size}_{seed}/basic_{dataset_name}_0-shot_litgpt/lm_tinyllama/")
        maybe_lora_paths = list(experiment_path.glob("lora_*"))
        if not maybe_lora_paths:
            continue
        lora_path = maybe_lora_paths[0] / ".cache/timing_lora.txt"
        predictions_path = experiment_path / "no_adaptation_bf16/.cache/timing_predictions.txt"
        calibration_path = experiment_path / "no_adaptation_bf16/affine_scalar/timing_calibration.txt"
        if lora_path.exists() and predictions_path.exists() and calibration_path.exists():
            with open(lora_path, "r") as f:
                timing_lora = float(f.read())
            with open(predictions_path, "r") as f:
                timing_predictions = float(f.read())
            with open(calibration_path, "r") as f:
                timing_calibration = float(f.read())
            data.append({
                "Dataset": dataset_name,
                "Samples\nper\nClass": size,
                "seed": seed,
                ("LoRA", ""): timing_lora,
                ("DP Calibration", "Forward pass"): timing_predictions, 
                ("DP Calibration", "Training"): timing_calibration,
                ("DP Calibration", "Total"): timing_predictions + timing_calibration,
                ("LoRA / DP Calibration", ""): timing_lora / (timing_predictions + timing_calibration)
            })
    # multicolum dataframe:
    df = pd.DataFrame(data)
    df = df.groupby(["Dataset", "Samples\nper\nClass"]).mean().drop(columns=["seed"])
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    # df = df.melt(id_vars=["dataset", "size", "seed"], var_name=["timing_type", "timing_subtype"], value_name="timing")
    # df = df.pivot_table(index=["dataset", "size", "seed"], columns=["timing_type", "timing_subtype"], values="timing")
    # df = df.reset_index()
    return df



# df = df.groupby(["dataset", "size"]).agg({
#     # "timing_lora": lambda x: f"{x.mean():.2f} ± {(x.std() if not np.isnan(x.std()) else 0.0):.2f}",
#     # "timing_predictions": lambda x: f"{x.mean():.2f} ± {(x.std() if not np.isnan(x.std()) else 0.0):.2f}",
#     # "affine_scalar": lambda x: f"{x.mean():.2f} ± {(x.std() if not np.isnan(x.std()) else 0.0):.2f}",
#     # "lora_over_calibration": lambda x: f"{x.mean():.2f} ± {(x.std() if not np.isnan(x.std()) else 0.0):.2f}"
#     "timing_lora": lambda x: f"{x.mean():.2f}",
#     "timing_predictions": lambda x: f"{x.mean():.2f}",
#     "affine_scalar": lambda x: f"{x.mean():.2f}",
#     "lora_over_calibration": lambda x: f"{x.mean():.2f}"
# })
# df.sort_values(by=["dataset", "size"]).reset_index(drop=True, inplace=True)
# df.to_latex


df = get_timings()
# df = df.groupby(["dataset", "size"]).mean()
df.round(2).to_latex("timings.tex", multirow=True, multicolumn=True, bold_rows=True,float_format="%d", column_format="llrrrrr")
df

LoRA DP Calibration             \
                                                 Forward pass   Training   
Dataset      Samples\nper\nClass                                           
20newsgroups 2                      365.421212      27.987083  15.785878   
             4                      249.327260      54.322011  15.042389   
             256                  11790.407426    3220.504969  24.889422   
agnews       4                      221.219308       4.815078  14.393258   
             8                      189.219729       7.183422  14.230722   
             256                    807.697042     153.553600  15.238711   
banking77    4                      754.611218     703.107154  15.536499   
             16                    1046.198218    2800.539820  18.684239   
             64                    7990.975716   11378.223075  28.210090   
dbpedia      2                      168.294151      14.818945  15.042994   
             4                      160.044970      27.558520  15.415924   
             256                   4694.298425    1599.192632  21.962570   
sst2         8                      230.112923       3.731867  13.483093   
             16                     190.433990       5.092996  14.112246   
             256                    741.298114      47.615787  14.418252   

                                               LoRA / DP Calibration  
                                         Total                        
Dataset      Samples\nper\nClass                                      
20newsgroups 2                       43.772961              8.348104  
             4                       69.364400              3.594456  
             256                   3245.394390              3.632966  
agnews       4                       19.208336             11.516839  
             8                       21.414145              8.836203  
             256                    168.792310              4.785153  
banking77    4                      718.643652              1.050049  
             16                    2819.224060              0.371094  
             64                   11406.433166              0.700567  
dbpedia      2                       29.861940              5.635741  
             4                       42.974444              3.724189  
             256                   1621.155202              2.895650  
sst2         8                       17.214960             13.367032  
             16                      19.205242              9.915730  
             256                     62.034040             11.949860